# Stock Markets Analytics Zoomcamp 2026 — Module 2 Homework: One Dataframe

Run each code cell, then fill in the **Answer** markdown cell below it.

Requirements: `pip install pandas numpy yfinance requests lxml gdown pyarrow`

Notes on data sources (confirmed against the actual course spec + lecture notebook):
- Q1 uses `iposcoop.com/ipos-recently-filed/` (the Withdrawn/Postponed list) —
  a **different** page from the lecture's `stockanalysis.com` IPO source.
- Q2/Q3 use `iposcoop.com/2025-pricings/` — also different from the lecture's
  `get_ipos_by_year()` (which pulls stockanalysis.com), per the homework spec.
- Q4 uses a separate precomputed parquet file (via `gdown`), with RSI as a
  **lowercase** `rsi` column — matching this course's TA-Lib convention.


In [1]:
import io
import re
import numpy as np
import pandas as pd
import requests
import yfinance as yf

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )
}


## Question 1: [IPO] Withdrawn IPOs by Company Type

What is the total withdrawn IPO value (in $ millions) for the company class
with the highest total withdrawal value?

- 200
- 300
- 400
- 500

Source: https://www.iposcoop.com/ipos-recently-filed/ — filter to
'Expected To Trade' == 'Withdrawn' (should be 32 entries).


In [2]:
url = "https://www.iposcoop.com/ipos-recently-filed/"
resp = requests.get(url, headers=HEADERS)
tables = pd.read_html(io.StringIO(resp.text))

ipo_df = None
for t in tables:
    if any("Expected" in str(c) for c in t.columns):
        ipo_df = t
        break

print("Columns found:", list(ipo_df.columns))
withdrawn = ipo_df[ipo_df["Expected To Trade"].astype(str).str.contains("Withdrawn", case=False, na=False)].copy()
print(f"Withdrawn entries: {len(withdrawn)}  (expected 32)")
withdrawn.head()


Columns found: ['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade', 'SCOOP Rating']
Withdrawn entries: 32  (expected 32)


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
6,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.0,NaN,NaN,$100.00,Withdrawn,S/O
12,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.0,$4.00,$5.00,$9.00,Withdrawn,S/O
21,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...,0.0,NaN,NaN,$100.00,Withdrawn,S/O
23,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.0,$4.00,$5.00,$22.50,Withdrawn,S/O
32,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.0,$10.00,$10.00,$200.00,Withdrawn,S/O


In [3]:
def classify_company(name):
    name = str(name)
    if "Technologies" in name:
        return "Technologies"
    if "Acquisition Corp" in name or "Acquisition Corporation" in name or "Corp" in name:
        return "Acquisition Corp"
    if "Inc" in name or "Incorporated" in name:
        return "Inc."
    if "Group" in name:
        return "Group"
    if "Ltd" in name or "Limited" in name:
        return "Limited"
    if "Holdings" in name or "Holding" in name:
        return "Holdings"
    return "Other"

name_col = [c for c in withdrawn.columns if "Company" in str(c)][0]
withdrawn["Company Type"] = withdrawn[name_col].apply(classify_company)

def parse_price(val):
    if pd.isna(val) or str(val).strip() in ("-", ""):
        return None
    nums = re.findall(r"[\d.]+", str(val))
    nums = [float(n) for n in nums]
    if not nums:
        return None
    return sum(nums) / len(nums)

price_col = [c for c in withdrawn.columns if "Price" in str(c)][0]
withdrawn["Avg_price"] = withdrawn[price_col].apply(parse_price)

def to_numeric_clean(val):
    if pd.isna(val) or str(val).strip() in ("-", ""):
        return np.nan
    cleaned = re.sub(r"[$,]", "", str(val)).strip()
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

shares_col = [c for c in withdrawn.columns if "Shares" in str(c)][0]
vol_col = [c for c in withdrawn.columns if "Vol" in str(c)][0]
withdrawn["Shares_millions"] = withdrawn[shares_col].apply(to_numeric_clean)
withdrawn["Est_Vol_millions"] = withdrawn[vol_col].apply(to_numeric_clean)

computed_value = withdrawn["Shares_millions"] * withdrawn["Avg_price"]
withdrawn["Shares_offered_value"] = computed_value.where(
    computed_value.notna(), withdrawn["Est_Vol_millions"]
)

result = withdrawn.groupby("Company Type")["Shares_offered_value"].sum().sort_values(ascending=False)
print(result)
print(f"\nHighest: {result.index[0]} with ${result.iloc[0]:.1f}M")


Company Type
Acquisition Corp    498.320
Inc.                338.000
Holdings            289.050
Other               274.795
Limited             194.500
Technologies        182.400
Group                29.000
Name: Shares_offered_value, dtype: float64

Highest: Acquisition Corp with $498.3M


### Answer 1
- Company type with highest total withdrawal value: Acquisition Corp
- Total value ($M): 500 (computed: $498.3M)


## Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

What is the median Sharpe ratio (as of 11 September 2026) for companies
that went public before 1 September 2025?

- -0.04
- 0.04
- 0.1
- 0.2

Source: https://www.iposcoop.com/2025-pricings/ — filter Offer Date <
2025-09-01, exclude 0% return (should leave 148 stocks; ~134 after
yfinance download, since some may be delisted).


In [4]:
url2 = "https://www.iposcoop.com/2025-pricings/"
resp2 = requests.get(url2, headers=HEADERS)
tables2 = pd.read_html(io.StringIO(resp2.text))

ipo_2025 = None
for t in tables2:
    if any("Offer" in str(c) for c in t.columns):
        ipo_2025 = t
        break

print("Columns:", list(ipo_2025.columns))
print(f"Total rows: {len(ipo_2025)}")


Columns: ['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)', 'Offer Price', '1st Day Close', 'Current Price', 'Return', 'SCOOP Rating']
Total rows: 231


In [5]:
date_col = [c for c in ipo_2025.columns if "Offer" in str(c) and "Date" in str(c)][0]
ipo_2025[date_col] = pd.to_datetime(ipo_2025[date_col], errors="coerce")

return_col = [c for c in ipo_2025.columns if "Return" in str(c) or "%" in str(c)]
return_col = return_col[0] if return_col else None

filtered = ipo_2025[ipo_2025[date_col] < "2025-09-01"].copy()
if return_col:
    filtered = filtered[filtered[return_col].astype(str).str.strip() != "0.00%"]

print(f"Filtered IPOs (before Sep 1 2025, non-zero return): {len(filtered)}  (expected ~148)")


Filtered IPOs (before Sep 1 2025, non-zero return): 146  (expected ~148)


In [8]:
ticker_col = [c for c in filtered.columns if "Symbol" in str(c) or "Ticker" in str(c)][0]
tickers = filtered[ticker_col].dropna().unique().tolist()
print(f"Tickers to download: {len(tickers)}")

all_data = []
failed = []
for tkr in tickers:
    try:
        data = yf.download(tkr, start="2024-01-01", end="2026-09-12",
                            progress=False, auto_adjust=False)
        if data.empty:
            failed.append(tkr)
            continue
        data = data.copy()
        # Flatten MultiIndex columns (field, ticker) -> just field, BEFORE any assignment
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        data["ticker"] = tkr
        all_data.append(data)
    except Exception:
        failed.append(tkr)

print(f"Successfully downloaded: {len(all_data)}  (expected ~134)")
print(f"Failed/delisted: {len(failed)}")

Tickers to download: 146


$MJID: possibly delisted; no timezone found

1 Failed download:
['MJID']: possibly delisted; no timezone found
$EMPG: possibly delisted; no timezone found

1 Failed download:
['EMPG']: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found

1 Failed download:
['CAEP']: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found

1 Failed download:
['PTNM']: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found

1 Failed download:
['AHL']: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found

1 Failed download:
['CEPT']: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found

1 Failed download:
['SDM']: possibly delisted; no timezone found
$NCT: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-12)

1 Failed download:
['NCT']: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-12)
$TBH: possibly delisted; no timezone found

1 Failed

Successfully downloaded: 131  (expected ~134)
Failed/delisted: 15


In [9]:
stocks_df = pd.concat(all_data).reset_index()

# Defensive check: confirm no duplicate columns survived
dupes = stocks_df.columns[stocks_df.columns.duplicated()]
print(f"Duplicate columns: {dupes.tolist()}")

Duplicate columns: []


In [11]:
stocks_df = stocks_df.sort_values(["ticker", "Date"])
stocks_df["growth_252d"] = stocks_df.groupby("ticker")["Close"].transform(lambda x: x / x.shift(252))
stocks_df["volatility"] = stocks_df.groupby("ticker")["Close"].transform(
    lambda x: x.rolling(30).std() * np.sqrt(252)
)

# Treat zero volatility as missing data (avoids division by zero -> inf)
stocks_df["volatility"] = stocks_df["volatility"].replace(0, np.nan)

stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

snapshot = stocks_df[stocks_df["Date"] == "2026-09-11"]
print(f"Stocks with data on 2026-09-11: {len(snapshot)}")
print(snapshot[["growth_252d", "volatility", "Sharpe"]].describe())

median_sharpe = snapshot["Sharpe"].median()
print(f"\nMEDIAN SHARPE RATIO: {median_sharpe:.3f}")

Stocks with data on 2026-09-11: 131
Price  growth_252d  volatility        Sharpe
count   129.000000  127.000000  1.270000e+02
mean      1.060208   23.497904  3.303727e+04
std       3.059704   41.474173  3.723091e+05
min       0.001005    0.000001 -4.014736e-02
25%       0.149038    2.377110  1.214157e-02
50%       0.596026    8.358907  4.800900e-02
75%       1.040233   23.968537  1.264168e-01
max      33.638270  276.424137  4.195710e+06

MEDIAN SHARPE RATIO: 0.048


### Answer 2

- Median Sharpe ratio (2026-09-11): 0.04 (computed: 0.048)


## Question 3: [IPO] 'Fixed Months Holding Strategy'

What is the optimal number of months (1 to 12) to hold a newly IPO'd stock
in order to maximize the median growth value?

- 1
- 3
- 5
- 7

Uses `stocks_df` from Question 2. 1 month = 21 trading days.


In [12]:
for m in range(1, 13):
    days = m * 21
    stocks_df[f"future_growth_{m}_m"] = stocks_df.groupby("ticker")["Close"].transform(
        lambda x: x.shift(-days) / x
    )

min_dates = stocks_df.groupby("ticker")["Date"].min().reset_index()
min_dates.columns = ["ticker", "min_date"]

entry_df = stocks_df.merge(min_dates, on="ticker").query("Date == min_date")

growth_cols = [f"future_growth_{m}_m" for m in range(1, 13)]
desc = entry_df[growth_cols].describe()
print(desc)

medians = desc.loc["50%"]
best_month = medians.idxmax()
print(f"\nBest holding period: {best_month} -> median growth {medians.max():.4f}")


       future_growth_1_m  future_growth_2_m  future_growth_3_m  \
count         131.000000         130.000000         130.000000   
mean            1.058606           1.179911           1.158919   
std             1.022669           1.738992           1.745652   
min             0.072941           0.106197           0.090147   
25%             0.709910           0.566315           0.439443   
50%             0.940080           0.891684           0.827951   
75%             1.096565           1.144196           1.094335   
max             8.787346          16.066783          16.344464   

       future_growth_4_m  future_growth_5_m  future_growth_6_m  \
count         130.000000         130.000000         130.000000   
mean            1.040069           0.995955           1.043197   
std             1.336805           1.223163           1.483950   
min             0.063927           0.032245           0.022075   
25%             0.376566           0.407421           0.315944   
50%      

### Answer 3
-  Optimal holding period (months): 1
-  Corresponding max median growth: 0.94 (computed: 0.9401, i.e. ~-6% median return)

## Question 4: [Strategy] Simple RSI-Based Trading Strategy

What is the total profit (in $ thousands) you would have earned by
investing $1000 every time a stock was oversold (RSI < 30)?

- 65
- 85
- 105
- 125

Uses a precomputed parquet file with technical/macro indicators for a
broad set of tickers, filtered 2000-01-01 to 2025-06-01. Note: RSI column
is lowercase `rsi`, matching this course's TA-Lib convention (confirmed
from the Module 2 lecture notebook's `talib_get_momentum_indicators_for_one_ticker`
function, which outputs a lowercase `rsi` column).


In [14]:
import gdown

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")
print(df.shape)
print(df.columns.tolist())


Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=c89701e6-0066-4673-8eb1-8c10356110a9
To: C:\Users\dmish\sma-zoomcamp\stock-markets-analytics-zoomcamp-2026\module-02-dataframe-analysis\notebooks\data.parquet
100%|██████████| 130M/130M [00:12<00:00, 10.3MB/s] 


(229932, 203)
['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'Year', 'Month', 'Weekday', 'Date', 'growth_1d', 'growth_3d', 'growth_7d', 'growth_30d', 'growth_90d', 'growth_365d', 'growth_future_30d', 'SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'is_positive_growth_30d_future', 'ticker_type', 'index_x', 'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc', 'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext', 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix', 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo', 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk', 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr', 'index_y', 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase', 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine', 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice', 'index', 

In [15]:
# Find the actual RSI and Date column names in this file (case can vary)
rsi_col = [c for c in df.columns if c.lower() == "rsi"][0]
date_col = "Date" if "Date" in df.columns else df.index.name
print(f"Using RSI column: '{rsi_col}', Date column/index: '{date_col}'")

if date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col])
else:
    df.index = pd.to_datetime(df.index)
    df = df.reset_index()
    date_col = df.columns[0]

selected_df = df[
    (df[rsi_col] < 30) &
    (df[date_col] >= "2000-01-01") &
    (df[date_col] <= "2025-06-01")
].copy()

print(f"Number of RSI<30 signals: {len(selected_df)}  (expected ~5,206)")

growth_col = [c for c in df.columns if "growth_future_30d" in c.lower()][0]
net_income = 1000 * (selected_df[growth_col] - 1).sum()
avg_return = (selected_df[growth_col] - 1).mean()
win_rate = (selected_df[growth_col] > 1).mean()

print(f"Net income: ${net_income:,.0f}  (${net_income/1000:.1f}K)")
print(f"Average 30-day return: {avg_return*100:.2f}%  (expected ~1.26%)")
print(f"Win rate: {win_rate*100:.2f}%  (expected ~55.13%)")


Using RSI column: 'rsi', Date column/index: 'Date'
Number of RSI<30 signals: 5206  (expected ~5,206)
Net income: $65,806  ($65.8K)
Average 30-day return: 1.26%  (expected ~1.26%)
Win rate: 55.13%  (expected ~55.13%)


RSI<30 signals: 5,206 (expected ~5,206 — exact match)
Average 30-day return: 1.26% (expected ~1.26% — exact match)
Win rate: 55.13% (expected ~55.13% — exact match)
Net income: $65,806 ($65.8K)

### Answer 4
-  Number of RSI<30 signals found: 5,206
-  Net income ($K): 65 (computed: $65.8K)

## Question 5 (Optional): Predicting a Positive-Return IPO

Most IPO strategies deliver negative average/median returns (even the 75th
percentile). How would you change the strategy to increase profitability?


### Answer 5

The current RSI<30 strategy is profitable but unfiltered — it buys every oversold signal indiscriminately, with a modest **55.13% win rate** and **1.26% average 30-day return**. A few concrete changes, based on patterns visible in this homework's own results, would likely improve profitability:

1. **Avoid combining the RSI strategy with newly-IPO'd stocks.** Q3 showed IPO'd stocks lose value on a median basis the longer they're held — even the best holding period (1 month) still had a median return of only **0.94** (a ~6% loss). Filtering out any ticker within its first 6–12 months of trading before applying the RSI<30 signal would likely raise the strategy's win rate, since much of the "oversold" signal on fresh IPOs may reflect genuine post-IPO price decay rather than a temporary dip worth buying.

2. **Add a trend/quality filter alongside RSI.** RSI<30 alone doesn't distinguish a temporary pullback in a fundamentally sound stock from a stock in genuine decline. Combining RSI<30 with a longer-term moving average filter (e.g., only buy if the stock is still above its **200-day moving average**) would filter out "falling knives" and keep only oversold dips within an otherwise healthy uptrend.

3. **Use volatility-adjusted position sizing instead of a flat $1,000.** Some tickers triggering RSI<30 are far more volatile than others (as seen in Q2's Sharpe ratio spread). Sizing positions inversely to recent volatility (smaller bets on high-volatility names, larger on stable ones) would reduce the impact of the worst outlier losses while keeping the average return intact.

4. **Tighten the RSI threshold back toward 25 instead of 30.** The homework's own observation notes that lowering the threshold from 30 to 25 last year produced fewer trades (**~1,568 vs. ~5,206**) — while the exact win-rate/return comparison isn't given, a stricter threshold generally selects for more genuinely oversold (rather than mildly weak) conditions, which is worth testing directly against this year's data to see if it trades off volume for a higher per-trade edge.

